# 🛡️ DeepGuard AI — V2 Training Pipeline (Cyclic Fine-Tuning)

**Click Runtime → Run all** and let it train automatically.

### What's different from V1:
1. ✅ **Loads existing best model** (`deepfake_model_best.keras`) from Google Drive
2. ✅ **Deduplication** — removes duplicate images using file hash before training
3. ✅ **Cyclic Training** — Phase 2 data is split into 6 cycles, each with shuffled mixed data
4. ✅ **Checkpoint per cycle** — saved to Drive after each cycle
5. ✅ **Fewer epochs per cycle** (3 epochs × 6 cycles = 18 effective epochs)
6. ✅ **Crash-safe** — resumes from last cycle checkpoint

> ⚠️ Make sure you're on a **GPU runtime**: Runtime → Change runtime type → T4 GPU


In [ ]:
# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================
import os, sys, warnings, time, hashlib
warnings.filterwarnings('ignore')
start_time = time.time()

# --- Check GPU ---
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU detected: {gpus[0].name}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print("✅ Mixed precision (fp16) enabled")
else:
    print("⚠️ No GPU detected! Training will be very slow.")
    print("   Go to Runtime → Change runtime type → T4 GPU")

# --- Mount Google Drive ---
from google.colab import drive
drive.mount('/content/drive')

DRIVE_OUTPUT = '/content/drive/MyDrive/DeepGuard_AI'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"✅ Output directory: {DRIVE_OUTPUT}")

# --- Setup Kaggle Credentials ---
os.environ['KAGGLE_USERNAME'] = 'YOUR_KAGGLE_USERNAME'
os.environ['KAGGLE_KEY'] = 'YOUR_KAGGLE_KEY'

import json
kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)
with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
    json.dump({'username': 'YOUR_KAGGLE_USERNAME', 'key': 'YOUR_KAGGLE_KEY'}, f)
os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)

# --- Install Dependencies ---
!pip install -q kaggle datasets mtcnn scikit-learn seaborn tqdm

print("\n✅ Environment setup complete!")

In [ ]:
# ============================================================
# 2. DOWNLOAD DATASETS
# ============================================================
import os

if not os.path.exists('/content/data_raw1'):
    print("📥 [1/4] Downloading Kaggle: deepfake-and-real-images...")
    !kaggle datasets download -d manjilkarki/deepfake-and-real-images --unzip -p /content/data_raw1
    print("   ✅ Done!")
else:
    print("✅ [1/4] Already downloaded: deepfake-and-real-images")

if not os.path.exists('/content/data_raw2'):
    print("📥 [2/4] Downloading Kaggle: real-and-fake-face-detection...")
    !kaggle datasets download -d ciplab/real-and-fake-face-detection --unzip -p /content/data_raw2
    print("   ✅ Done!")
else:
    print("✅ [2/4] Already downloaded: real-and-fake-face-detection")

if not os.path.exists('/content/data_raw3'):
    print("📥 [3/4] Downloading Kaggle: 140k-real-and-fake-faces...")
    !kaggle datasets download -d xhlulu/140k-real-and-fake-faces --unzip -p /content/data_raw3
    print("   ✅ Done!")
else:
    print("✅ [3/4] Already downloaded: 140k-real-and-fake-faces")

if not os.path.exists('/content/data_raw4'):
    print("📥 [4/4] Downloading Kaggle: cifake-real-and-ai-generated-synthetic-images...")
    !kaggle datasets download -d birdy654/cifake-real-and-ai-generated-synthetic-images --unzip -p /content/data_raw4
    print("   ✅ Done!")
else:
    print("✅ [4/4] Already downloaded: cifake-real-and-ai-generated-synthetic-images")

print("\n✅ All dataset downloads complete!")

In [ ]:
# ============================================================
# 3. ORGANIZE DATA + DEDUPLICATE (removes exact duplicate images)
# ============================================================
import shutil, glob, os, hashlib
from tqdm import tqdm

DATA_DIR = '/content/data'
real_dir = f'{DATA_DIR}/real'
fake_dir = f'{DATA_DIR}/fake'
os.makedirs(real_dir, exist_ok=True)
os.makedirs(fake_dir, exist_ok=True)

def safe_copy(src_pattern, dest_dir, label=""):
    files = glob.glob(src_pattern)
    copied = 0
    for f in files:
        if os.path.isfile(f):
            try:
                shutil.copy2(f, dest_dir)
                copied += 1
            except:
                pass
    if copied > 0:
        print(f"   Copied {copied} {label} files")
    return copied

# --- Kaggle Dataset 1 ---
print("📂 Processing Kaggle Dataset 1...")
for base in ['/content/data_raw1', '/content/data_raw1/Dataset']:
    for split in ['', 'Train', 'Test', 'Validation']:
        path = os.path.join(base, split) if split else base
        for real_name in ['Real', 'real']:
            safe_copy(os.path.join(path, real_name, '*'), real_dir, f"real from {path}")
        for fake_name in ['Fake', 'fake']:
            safe_copy(os.path.join(path, fake_name, '*'), fake_dir, f"fake from {path}")

# --- Kaggle Dataset 2 ---
print("📂 Processing Kaggle Dataset 2...")
for base in ['/content/data_raw2', '/content/data_raw2/real_and_fake_face']:
    safe_copy(os.path.join(base, 'training_real', '*'), real_dir, "real")
    safe_copy(os.path.join(base, 'training_fake', '*'), fake_dir, "fake")

# --- Kaggle Dataset 3 ---
print("📂 Processing Kaggle Dataset 3 (140k faces)...")
for base in ['/content/data_raw3', '/content/data_raw3/real_and_fake_face', '/content/data_raw3/140k-real-and-fake-faces']:
    for split in ['', 'train', 'val', 'test', 'valid', 'real_and_fake_face']:
        path = os.path.join(base, split) if split else base
        for real_name in ['real', 'Real', 'training_real']:
            safe_copy(os.path.join(path, real_name, '*'), real_dir, "real")
        for fake_name in ['fake', 'Fake', 'training_fake']:
            safe_copy(os.path.join(path, fake_name, '*'), fake_dir, "fake")

# --- Kaggle Dataset 4 (GenAI / Diffusion) ---
print("📂 Processing Kaggle Dataset 4 (GenAI faces)...")
for base in ['/content/data_raw4', '/content/data_raw4/real_vs_ai', '/content/data_raw4/Data']:
    for split in ['', 'train', 'val', 'test', 'Data']:
        path = os.path.join(base, split) if split else base
        for real_name in ['real', 'Real', 'REAL', 'training_real']:
            safe_copy(os.path.join(path, real_name, '*'), real_dir, "real GenAI")
        for fake_name in ['fake', 'Fake', 'FAKE', 'ai', 'AI', 'ai_generated', 'training_fake']:
            safe_copy(os.path.join(path, fake_name, '*'), fake_dir, "fake GenAI")

print(f"\n   Before dedup: Real={len(os.listdir(real_dir))}, Fake={len(os.listdir(fake_dir))}")

# ---- DEDUPLICATION: Remove duplicate images by file hash ----
print("\n🔍 Deduplicating images (removing exact duplicates)...")

def deduplicate_dir(directory):
    """Remove duplicate files by MD5 hash. Returns count of removed files."""
    seen_hashes = set()
    removed = 0
    files = sorted(os.listdir(directory))
    for fname in tqdm(files, desc=f"  Dedup {os.path.basename(directory)}"):
        fpath = os.path.join(directory, fname)
        if not os.path.isfile(fpath):
            continue
        # Hash first 8KB for speed (enough to detect duplicates)
        with open(fpath, 'rb') as f:
            file_hash = hashlib.md5(f.read(8192)).hexdigest()
        if file_hash in seen_hashes:
            os.remove(fpath)
            removed += 1
        else:
            seen_hashes.add(file_hash)
    return removed

removed_real = deduplicate_dir(real_dir)
removed_fake = deduplicate_dir(fake_dir)

real_count = len(os.listdir(real_dir))
fake_count = len(os.listdir(fake_dir))
print(f"\n   Removed {removed_real} duplicate real + {removed_fake} duplicate fake images")
print(f"\n📊 FINAL DEDUPLICATED DATASET:")
print(f"   Real images: {real_count:,}")
print(f"   Fake images: {fake_count:,}")
print(f"   Total:       {real_count + fake_count:,}")
print("\n✅ Data organization + deduplication complete!")

In [ ]:
# ============================================================
# 4. HYPERPARAMETERS
# ============================================================
import tensorflow as tf
from tensorflow.keras import layers

IMG_SIZE = 380
BATCH_SIZE = 16
LR_PHASE2 = 1e-5
EPOCHS_PER_CYCLE = 3       # 3 epochs per cycle
NUM_CYCLES = 6             # 6 cycles = 18 effective epochs
DROPOUT_1 = 0.5
DROPOUT_2 = 0.3
DENSE_UNITS = 256
UNFREEZE_LAYERS = 50
EARLY_STOP_PATIENCE = 2    # per cycle (stops a bad cycle early)

print(f"📋 Training Plan:")
print(f"   {NUM_CYCLES} cycles × {EPOCHS_PER_CYCLE} epochs = {NUM_CYCLES * EPOCHS_PER_CYCLE} total epochs")
print(f"   Each cycle uses 1/{NUM_CYCLES} of the data (shuffled, no overlap)")
print(f"   Checkpoint saved after every cycle")

In [ ]:
# ============================================================
# 5. LOAD EXISTING BEST MODEL FROM GOOGLE DRIVE
# ============================================================
import os

# Priority: best > regular deepfake_model
MODEL_CANDIDATES = [
    f'{DRIVE_OUTPUT}/deepfake_model_best.keras',
    f'{DRIVE_OUTPUT}/deepfake_model.keras',
]

model = None
loaded_from = None

for candidate in MODEL_CANDIDATES:
    if os.path.exists(candidate):
        try:
            model = tf.keras.models.load_model(candidate, compile=False)
            loaded_from = candidate
            size_mb = os.path.getsize(candidate) / (1024*1024)
            print(f"✅ Loaded model: {candidate} ({size_mb:.1f} MB)")
            print(f"   Input shape: {model.input_shape}")
            print(f"   Output shape: {model.output_shape}")
            break
        except Exception as e:
            print(f"⚠️ Failed to load {candidate}: {e}")

if model is None:
    print("❌ No existing model found! Cannot continue.")
    print(f"   Looked in: {MODEL_CANDIDATES}")
    raise FileNotFoundError("Upload deepfake_model_best.keras to Google Drive/DeepGuard_AI/")

# Get base_model reference for unfreezing
base_model = model.get_layer('efficientnetb4')
print(f"\n   Base model: {base_model.name} ({len(base_model.layers)} layers)")

# Save class mapping
class_names = ['fake', 'real']  # alphabetical order from image_dataset_from_directory
print(f"   Classes: {class_names}")
print(f"   Label mapping: 0={class_names[0]}, 1={class_names[1]}")

In [ ]:
# ============================================================
# 6. PREPARE CYCLIC DATA SPLITS (shuffle + split into N cycles)
# ============================================================
import numpy as np
import random

# Get all file paths
real_files = sorted([os.path.join(real_dir, f) for f in os.listdir(real_dir) if os.path.isfile(os.path.join(real_dir, f))])
fake_files = sorted([os.path.join(fake_dir, f) for f in os.listdir(fake_dir) if os.path.isfile(os.path.join(fake_dir, f))])

# Shuffle with fixed seed for reproducibility
random.seed(42)
random.shuffle(real_files)
random.shuffle(fake_files)

# Split into NUM_CYCLES non-overlapping chunks
def split_into_chunks(file_list, n_chunks):
    chunk_size = len(file_list) // n_chunks
    chunks = []
    for i in range(n_chunks):
        start = i * chunk_size
        end = start + chunk_size if i < n_chunks - 1 else len(file_list)
        chunks.append(file_list[start:end])
    return chunks

real_chunks = split_into_chunks(real_files, NUM_CYCLES)
fake_chunks = split_into_chunks(fake_files, NUM_CYCLES)

print(f"📊 Data Split into {NUM_CYCLES} Cycles:")
print(f"   Total: {len(real_files):,} real + {len(fake_files):,} fake")
for i in range(NUM_CYCLES):
    print(f"   Cycle {i+1}: {len(real_chunks[i]):,} real + {len(fake_chunks[i]):,} fake = {len(real_chunks[i]) + len(fake_chunks[i]):,} images")

print("\n✅ No image appears in more than one cycle (zero overlap)!")

In [ ]:
# ============================================================
# 7. CYCLIC FINE-TUNING (Phase 2 — 6 cycles × 3 epochs each)
# ============================================================
from tensorflow.keras import callbacks
import shutil, time

# Unfreeze top layers of backbone
base_model.trainable = True
for layer in base_model.layers[:-UNFREEZE_LAYERS]:
    layer.trainable = False

trainable_count = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
frozen_count = model.count_params() - trainable_count

print("=" * 60)
print("🚀 PHASE 2: Cyclic Fine-Tuning")
print("=" * 60)
print(f"   Unfrozen layers: last {UNFREEZE_LAYERS}")
print(f"   Trainable params: {trainable_count:,}")
print(f"   Frozen params:    {frozen_count:,}")
print(f"   Plan: {NUM_CYCLES} cycles × {EPOCHS_PER_CYCLE} epochs")
print()

# Compile
model.compile(
    optimizer=tf.keras.optimizers.AdamW(
        learning_rate=LR_PHASE2,
        weight_decay=1e-4
    ),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.AUC(name='auc'),
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

all_histories = []
best_val_auc = 0.0

# Check which cycle to resume from
start_cycle = 0
for c in range(NUM_CYCLES - 1, -1, -1):
    ckpt = f'{DRIVE_OUTPUT}/checkpoint_cycle_{c+1}.keras'
    if os.path.exists(ckpt):
        try:
            model = tf.keras.models.load_model(ckpt, compile=False)
            model.compile(
                optimizer=tf.keras.optimizers.AdamW(learning_rate=LR_PHASE2, weight_decay=1e-4),
                loss='binary_crossentropy',
                metrics=['accuracy', tf.keras.metrics.AUC(name='auc'),
                         tf.keras.metrics.Precision(name='precision'),
                         tf.keras.metrics.Recall(name='recall')]
            )
            # Re-unfreeze since loaded model may have different trainable flags
            base_model = model.get_layer('efficientnetb4')
            base_model.trainable = True
            for layer in base_model.layers[:-UNFREEZE_LAYERS]:
                layer.trainable = False
            start_cycle = c + 1
            print(f"🔄 Resuming from cycle {c+1} checkpoint: {ckpt}")
            break
        except:
            pass

if start_cycle > 0:
    print(f"   Skipping cycles 1-{start_cycle}, starting from cycle {start_cycle + 1}")
else:
    print("   Starting from cycle 1 (no checkpoints found)")

# ---- RUN CYCLIC TRAINING ----
for cycle_idx in range(start_cycle, NUM_CYCLES):
    cycle_num = cycle_idx + 1
    print(f"\n{'='*60}")
    print(f"🔄 CYCLE {cycle_num}/{NUM_CYCLES}")
    print(f"{'='*60}")

    # Create temp directory for this cycle's data
    cycle_data_dir = f'/content/cycle_data'
    cycle_real = f'{cycle_data_dir}/real'
    cycle_fake = f'{cycle_data_dir}/fake'

    # Clean previous cycle data
    if os.path.exists(cycle_data_dir):
        shutil.rmtree(cycle_data_dir)
    os.makedirs(cycle_real, exist_ok=True)
    os.makedirs(cycle_fake, exist_ok=True)

    # Symlink this cycle's files (fast, no copy)
    for f in real_chunks[cycle_idx]:
        dst = os.path.join(cycle_real, os.path.basename(f))
        if not os.path.exists(dst):
            os.symlink(f, dst)
    for f in fake_chunks[cycle_idx]:
        dst = os.path.join(cycle_fake, os.path.basename(f))
        if not os.path.exists(dst):
            os.symlink(f, dst)

    cycle_real_count = len(os.listdir(cycle_real))
    cycle_fake_count = len(os.listdir(cycle_fake))
    print(f"   Data: {cycle_real_count:,} real + {cycle_fake_count:,} fake")

    # Build tf.data pipeline for this cycle
    cycle_train_ds = tf.keras.utils.image_dataset_from_directory(
        cycle_data_dir,
        validation_split=0.15,
        subset='training',
        seed=42 + cycle_idx,  # Different seed per cycle for varied splits
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='binary'
    )
    cycle_val_ds = tf.keras.utils.image_dataset_from_directory(
        cycle_data_dir,
        validation_split=0.15,
        subset='validation',
        seed=42 + cycle_idx,
        image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        label_mode='binary'
    )

    AUTOTUNE = tf.data.AUTOTUNE
    cycle_train_ds = cycle_train_ds.prefetch(buffer_size=AUTOTUNE)
    cycle_val_ds = cycle_val_ds.prefetch(buffer_size=AUTOTUNE)

    # Callbacks for this cycle
    cycle_ckpt_path = f'{DRIVE_OUTPUT}/checkpoint_cycle_{cycle_num}.keras'
    best_ckpt_path = f'{DRIVE_OUTPUT}/deepfake_model_best.keras'

    cycle_callbacks = [
        callbacks.EarlyStopping(
            monitor='val_auc', patience=EARLY_STOP_PATIENCE,
            restore_best_weights=True, mode='max', verbose=1
        ),
        callbacks.ModelCheckpoint(
            cycle_ckpt_path,
            save_best_only=True,
            monitor='val_auc', mode='max', verbose=1
        ),
        callbacks.ModelCheckpoint(
            best_ckpt_path,
            save_best_only=True,
            monitor='val_auc', mode='max', verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_auc', factor=0.5, patience=1,
            mode='max', min_lr=1e-7, verbose=1
        )
    ]

    # Train this cycle
    cycle_start = time.time()
    history = model.fit(
        cycle_train_ds,
        validation_data=cycle_val_ds,
        epochs=EPOCHS_PER_CYCLE,
        callbacks=cycle_callbacks,
        verbose=1
    )
    cycle_time = time.time() - cycle_start
    all_histories.append(history)

    # Report
    cycle_best_auc = max(history.history.get('val_auc', [0]))
    cycle_best_acc = max(history.history.get('val_accuracy', [0]))
    if cycle_best_auc > best_val_auc:
        best_val_auc = cycle_best_auc

    print(f"\n   ✅ Cycle {cycle_num} done in {cycle_time/60:.1f} min")
    print(f"   📊 Best Val AUC this cycle: {cycle_best_auc:.4f}")
    print(f"   📊 Best Val Acc this cycle: {cycle_best_acc:.4f}")
    print(f"   📊 Overall Best Val AUC:    {best_val_auc:.4f}")
    print(f"   💾 Saved: {cycle_ckpt_path}")

    # Clean up cycle data to free disk
    shutil.rmtree(cycle_data_dir, ignore_errors=True)

total_time = time.time() - start_time
print(f"\n{'='*60}")
print(f"🎉 ALL {NUM_CYCLES} CYCLES COMPLETE!")
print(f"{'='*60}")
print(f"   Total time: {total_time/60:.1f} minutes")
print(f"   Best Val AUC: {best_val_auc:.4f}")
print(f"   Model saved to: {best_ckpt_path}")

In [ ]:
# ============================================================
# 8. EVALUATION ON FULL VALIDATION SET
# ============================================================
import numpy as np
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_curve, auc as sklearn_auc, precision_recall_curve
)
import seaborn as sns
import matplotlib.pyplot as plt

# Build a fresh val dataset from ALL data for final evaluation
print("Building full validation dataset for final evaluation...")
full_val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset='validation',
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
).prefetch(tf.data.AUTOTUNE)

print("Running evaluation...")
y_true = []
y_pred_proba = []

for images, labels in full_val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_pred_proba.extend(preds.flatten())

y_true = np.array(y_true)
y_pred_proba = np.array(y_pred_proba)
y_pred = (y_pred_proba > 0.5).astype(int)

# Classification Report
print("\n" + "=" * 60)
print("📋 CLASSIFICATION REPORT")
print("=" * 60)
report = classification_report(y_true, y_pred, target_names=class_names)
print(report)

# Plots
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

cm = confusion_matrix(y_true, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=class_names, yticklabels=class_names, annot_kws={'size': 16})
axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted', fontsize=12)
axes[0].set_ylabel('Actual', fontsize=12)

fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc_score = sklearn_auc(fpr, tpr)
axes[1].plot(fpr, tpr, 'b-', linewidth=2.5, label=f'AUC = {roc_auc_score:.4f}')
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5)
axes[1].fill_between(fpr, tpr, alpha=0.1, color='blue')
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].legend(fontsize=13, loc='lower right')
axes[1].grid(True, alpha=0.3)

precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_proba)
pr_auc = sklearn_auc(recall_vals, precision_vals)
axes[2].plot(recall_vals, precision_vals, 'g-', linewidth=2.5, label=f'PR-AUC = {pr_auc:.4f}')
axes[2].fill_between(recall_vals, precision_vals, alpha=0.1, color='green')
axes[2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
axes[2].set_xlabel('Recall', fontsize=12)
axes[2].set_ylabel('Precision', fontsize=12)
axes[2].legend(fontsize=13, loc='lower left')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{DRIVE_OUTPUT}/evaluation_results_v2.png', dpi=150, bbox_inches='tight')
plt.show()

# Optimal threshold
optimal_idx = np.argmax(tpr - fpr)
optimal_threshold = float(thresholds[optimal_idx])

accuracy = float(np.mean(y_pred == y_true))
metrics_dict = {
    'accuracy': accuracy,
    'auc_roc': float(roc_auc_score),
    'pr_auc': float(pr_auc),
    'optimal_threshold': optimal_threshold,
    'total_val_samples': int(len(y_true)),
    'class_names': class_names,
    'confusion_matrix': cm.tolist(),
    'training_cycles': NUM_CYCLES,
    'epochs_per_cycle': EPOCHS_PER_CYCLE,
    'training_time_minutes': round(total_time / 60, 1)
}

with open(f'{DRIVE_OUTPUT}/metrics_v2.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)

print(f"\n🎯 Final Metrics:")
print(f"   Accuracy:           {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   AUC-ROC:            {roc_auc_score:.4f}")
print(f"   PR-AUC:             {pr_auc:.4f}")
print(f"   Optimal Threshold:  {optimal_threshold:.4f}")
print(f"\n✅ Evaluation complete!")

In [ ]:
# ============================================================
# 9. SAVE FINAL MODEL
# ============================================================
import os

print("=" * 60)
print("💾 SAVING FINAL MODEL")
print("=" * 60)

# Save as deepfake_model.keras (the main deployment model)
model.save(f'{DRIVE_OUTPUT}/deepfake_model.keras')
print(f"✅ Saved: deepfake_model.keras")

# Also ensure best is saved
model.save(f'{DRIVE_OUTPUT}/deepfake_model_best.keras')
print(f"✅ Saved: deepfake_model_best.keras")

# TFLite
try:
    converter = tf.lite.TFLiteConverter.from_keras_model(model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]
    tflite_model = converter.convert()
    with open(f'{DRIVE_OUTPUT}/deepfake_model.tflite', 'wb') as f:
        f.write(tflite_model)
    print(f"✅ Saved: deepfake_model.tflite")
except Exception as e:
    print(f"⚠️ TFLite conversion failed: {e}")

# List all saved files
print(f"\n📁 All outputs in Google Drive: {DRIVE_OUTPUT}")
print("-" * 50)
total_size = 0
for f in sorted(os.listdir(DRIVE_OUTPUT)):
    path = os.path.join(DRIVE_OUTPUT, f)
    if os.path.isfile(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        total_size += size_mb
        print(f"  📄 {f:<40} {size_mb:>8.1f} MB")
print(f"  {'─' * 49}")
print(f"  {'Total':<40} {total_size:>8.1f} MB")

total_time_final = time.time() - start_time
print(f"\n⏱️  Total pipeline time: {total_time_final/60:.1f} minutes")
print(f"\n{'=' * 60}")
print("🎉 V2 TRAINING PIPELINE COMPLETE!")
print("=" * 60)
print(f"\nNext steps:")
print(f"  1. Go to Google Drive → DeepGuard_AI folder")
print(f"  2. Download 'deepfake_model_best.keras'")
print(f"  3. Place it in your project as 'streamlit/deepfake_model_best.keras'")
print(f"  4. Run: python app.py")
print(f"  5. Open http://localhost:7860")